# 02 — Preprocessing: Textbereinigung & Normalisierung

**Ziel dieses Notebooks:** Den bereinigten Datensatz (`train_clean.csv`,
7.434 Zeilen) fuer die Feature-Extraktion vorbereiten — Rohtext in eine
fuer TF-IDF/BoW geeignete Form ueberfuehren, ohne vorschnell Information
zu verlieren.

**Leitfragen:**
- Wie wird der Text strukturell bereinigt (URLs, Encoding, HTML-Entities)?
- Wort-Tokenisierung: ausreichend, oder waere Subword-Tokenisierung
  (Bonus-Kandidat) relevant?
- Stemming vs. Lemmatisierung: welche Unterschiede zeigen sich konkret?
- Erste Indikatoren zur Aufarbeitungs-Qualitaet (Vokabulargroesse,
  Entropie, Chi2 roh vs. bereinigt)

**Wichtige Einschraenkung:** Die hier gemessenen Indikatoren (Zelle 12/13)
sind VORLAEUFIG. Der belastbare Beweis, ob eine Preprocessing-Variante
tatsaechlich hilft, erfolgt erst in Phase 4 als Ablationsstudie (gleiches
Modell, verschiedene Text-Varianten, F1-Vergleich per Cross-Validation).

**Prinzip:** Jede Transformation wird als neue Spalte angehaengt
(tokens_stemmed, tokens_lemmatized, ...), nichts wird ueberschrieben —
ermoeglicht spaeteren direkten Vergleich.

**Output:** `src/preprocessing.py` (importierbare Cleaning-Funktionen),
`data/processed/train_preprocessed.csv`, Plots/Tabellen wie gewohnt.

In [ ]:
# =============================================================================
# Zelle 02 – Imports, Store44-Style aktivieren, bereinigten Datensatz laden
# =============================================================================

import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from viz_config import (
    apply_store44_style, save_figure,
    COLOR_GOLD, COLOR_BLUE, COLOR_GREEN,
    COLOR_CLASS_0, COLOR_CLASS_1, COLOR_TEXT_MUTED
)

apply_store44_style()

# Bereinigter Datensatz aus Notebook 01 (Duplikate/Konflikte bereits entfernt)
train = pd.read_csv("../data/processed/train_clean.csv")

print("Shape:", train.shape)
print("Spalten:", list(train.columns))
train[["text", "target"]].head()

In [ ]:
# =============================================================================
# Zelle 03 (final) – Cleaning-Funktion: URLs, Mojibake, Zahlen-Platzhalter,
# HTML-Entities, lowercase
# =============================================================================

import re
import html

def clean_text(text):
    text = html.unescape(text)
    text = re.sub(r"http[s]?://\S+", "", text)           # URLs entfernen
    text = re.sub(r"[\x80-\xff]", "", text)               # Mojibake entfernen
    text = re.sub(r"\d+(?:[.,]\d+)*", "NUM", text)         # Zahlen -> Platzhalter
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

train["text_clean"] = train["text"].apply(clean_text)

print("=== Test: Zahlen-Platzhalter ===")
print(clean_text("13,000 people receive #wildfires evacuation orders"))
print(clean_text("Magnitude 1.4 earthquake reported"))

print("\n=== Vorher/Nachher-Beispiele ===\n")
for i in [0, 2, 3]:
    print(f"VORHER:  {train['text'].iloc[i]}")
    print(f"NACHHER: {train['text_clean'].iloc[i]}\n")

print("=== Encoding-Artefakt-Beispiel ===")
encoding_example = train[train["has_encoding_artifact"]].iloc[0]
print(f"VORHER:  {encoding_example['text']}")
print(f"NACHHER: {clean_text(encoding_example['text'])}")

In [ ]:
# =============================================================================
# Zelle 03b – Bereinigung auf text_clean-Basis: Duplikate/Konflikte entfernen
# =============================================================================
# Ergaenzung zur Notebook-01-Bereinigung (die nur auf Roh-Text prueft):
# URL-Entfernung/Lowercase/NUM-Normalisierung erzeugt weitere 70 Konflikt-
# Gruppen (253 Zeilen). Bevor Tokenisierung/Stopwords/Stemming/Lemmatisierung
# auf diesen Zeilen aufgebaut werden, werden sie hier entfernt.
# Begruendung und Analyse: Notebook 01, Zelle 03b + Zelle 04.

print("Shape vor Bereinigung auf text_clean-Basis:", train.shape)

# Flag 1: text_clean-Konflikte (widersprüchliche Labels nach Normalisierung)
label_var_clean = train.groupby("text_clean")["target"].nunique()
conflict_texts_clean = label_var_clean[label_var_clean > 1].index
train["is_label_conflict_clean"] = train["text_clean"].isin(conflict_texts_clean)

# Flag 2: harmlose text_clean-Duplikate (gleiche Labels, aber identisch nach Normalisierung)
train["is_exact_duplicate_clean"] = (
    train.duplicated(subset=["text_clean", "target"], keep="first") &
    ~train["is_label_conflict_clean"]
)

print(f"\nNeu identifizierte Konflikt-Zeilen: {train['is_label_conflict_clean'].sum()}")
print(f"Neu identifizierte harmlose Duplikat-Zeilen: {train['is_exact_duplicate_clean'].sum()}")

# Bereinigung
train_clean2 = train[
    ~train["is_label_conflict_clean"] &
    ~train["is_exact_duplicate_clean"]
].copy()

# Hilfs-Flags entfernen
train_clean2 = train_clean2.drop(
    columns=["is_label_conflict_clean", "is_exact_duplicate_clean"]
)

print(f"\nShape nach Bereinigung: {train_clean2.shape}")
print(f"Entfernte Zeilen: {train.shape[0] - train_clean2.shape[0]}")
print(f"\nNeue Klassenverteilung:")
print(train_clean2["target"].value_counts(normalize=True).round(3))

# Ab hier arbeiten wir mit train_clean2 weiter
train = train_clean2.copy()

## Erkenntnis: Textbereinigung

**Durchgefuehrte Transformationen** (alle in `text_clean`, Original
`text` bleibt erhalten):
- HTML-Entities dekodiert (&amp; -> &)
- URLs entfernt (Inhalt war Noise, siehe Notebook 01)
- Mojibake-Zeichen entfernt (\x80-\xff, vollstaendiger Bereich statt
  urspruenglich zu eng gefasstem Pattern — korrigiert nach Stichproben-
  pruefung, 9,2% der Tweets betroffen, IDENTISCH in Kaggle- und Schul-
  version, also Datensatz-inhärent)
- Zahlen durch Platzhalter "NUM" ersetzt (exakte Werte generalisieren
  nicht, analog zur URL-Logik; im Gegensatz zu URLs bleibt hier ABER ein
  Token im Text, daher KEINE zusaetzliche has_number-Spalte noetig —
  der Vectorizer in Phase 3 erfasst die NUM-Haeufigkeit automatisch)
- Lowercase

**Bewusst NICHT behandelt** (uebernimmt der Standard-Tokenizer in Phase 3
automatisch, siehe Test in Zelle 03):
- `#` wird entfernt, Hashtag-Wort verschmilzt korrekt mit gleichem
  Wort ohne Hashtag
- `.` und sonstige Satzzeichen werden entfernt
- Apostrophe brechen Kontraktionen (z. B. "don't" -> "don"+"t") —
  geringer, akzeptierter Informationsverlust, Standard-Verhalten bei
  Bag-of-Words-Modellen

In [ ]:
# =============================================================================
# Zelle 05 – Tokenisierung (Wort-Ebene)
# =============================================================================
# Nutzt denselben Tokenizer-Ansatz wie der spaetere sklearn-Vectorizer
# (Konsistenz zwischen Preprocessing-Anzeige und tatsaechlicher Vektorisierung),
# hier aber explizit als eigene Spalte zur Weiterverarbeitung (Stopwords,
# Stemming, Lemmatisierung).

import re

def tokenize(text):
    return re.findall(r"\b[a-z]+\b", text)

train["tokens"] = train["text_clean"].apply(tokenize)

# Kontrolle
print("=== Beispiele ===\n")
for i in [0, 2, 3]:
    print(f"TEXT:   {train['text_clean'].iloc[i]}")
    print(f"TOKENS: {train['tokens'].iloc[i]}\n")

avg_tokens = train["tokens"].apply(len).mean()
print(f"Durchschnittliche Anzahl Tokens pro Tweet: {avg_tokens:.1f}")

## Erkenntnis: Tokenisierungsgroesse — Wort- vs. Subword-Ebene

**Gewaehlt: Wort-Tokenisierung** (jedes Wort = ein Token).

**Alternative: Subword-Tokenisierung** (z. B. Byte-Pair-Encoding/BPE,
WordPiece) — zerlegt Woerter in haeufige Teilstuecke, z. B.
"unbelievable" -> "un" + "believ" + "able". Standard bei Transformern
(BERT nutzt WordPiece, GPT nutzt BPE).

**Warum Wort-Ebene fuer das Standardprojekt:**
- TF-IDF/BoW basieren konzeptionell auf Wort-Haeufigkeiten — Subword-
  Tokenisierung ergibt hier keinen Mehrwert, da das Modell ohnehin keine
  Wortteile sinnvoll kombiniert (im Gegensatz zu neuronalen Netzen mit
  gelernten Embeddings)
- Subword-Tokenisierung erfordert ein trainiertes Tokenizer-Vokabular
  (z. B. via `tokenizers`-Bibliothek oder vortrainierte Transformer-
  Tokenizer) — zusaetzliche Komplexitaet ohne Nutzen fuer sparse Modelle

**Warum Subword-Tokenisierung relevant fuer den Bonus-Branch:**
- Robuster gegen Tippfehler/seltene Wortformen (z. B. "wildfires" und
  "wildfire" teilen sich Subword-Einheiten, IST-Vokabular bleibt klein)
- Notwendig fuer vortrainierte Transformer-Modelle (BERT/DistilBERT
  erwarten exakt ihr eigenes Subword-Vokabular)
- Erwartete Messung des Effekts (Bonus-Phase): Vokabulargroesse
  Wort- vs. Subword-Tokenisierung im direkten Vergleich, sowie Einfluss
  auf OOV-Rate (Out-of-Vocabulary) bei seltenen/falsch geschriebenen
  Woertern

**Konsequenz:** Wort-Tokenisierung bleibt fuer Phase 2-7 verbindlich.
Subword-Tokenisierung wird im Bonus-Branch (Phase 8) eingefuehrt und
explizit gegen die Wort-Ebene verglichen.

In [ ]:
# =============================================================================
# Zelle 07 – Stopword-Entfernung
# =============================================================================

import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

def remove_stopwords(tokens):
    return [t for t in tokens if t not in stop_words]

train["tokens_no_stopwords"] = train["tokens"].apply(remove_stopwords)

# Kontrolle
print("=== Beispiele ===\n")
for i in [0, 1, 2]:
    print(f"VORHER:  {train['tokens'].iloc[i]}")
    print(f"NACHHER: {train['tokens_no_stopwords'].iloc[i]}\n")

avg_before = train["tokens"].apply(len).mean()
avg_after = train["tokens_no_stopwords"].apply(len).mean()
print(f"Durchschnittliche Tokenanzahl vorher: {avg_before:.1f}")
print(f"Durchschnittliche Tokenanzahl nachher: {avg_after:.1f}")
print(f"Reduktion: {(1 - avg_after/avg_before)*100:.1f}%")

## Erkenntnis: Stopword-Entfernung

Durchschnittliche Tokenanzahl: 14,3 -> 9,2 (-35,3%). Erwartbare
Groessenordnung fuer englischen Text.

**Hinweis aus Notebook 01 (Zelle 09c) bestaetigt sich:** Nur 3,4% der
urspruenglichen Vokabular-Ueberschneidung zwischen den Klassen waren
Stopwords — die Reduktion hier betrifft also primaer haeufige,
thematisch neutrale Fuellwoerter (the, of, in, to), nicht die fuer die
Klassifikation eigentlich relevanten Vokabular-Unterschiede.

**Erwartung fuer Phase 3:** Vokabular sollte durch Stopword-Entfernung
spuerbar kleiner werden (weniger Dimensionen im TF-IDF-Vektor), bei
nahezu gleichbleibender Trennschaerfe — wird in Zelle 12/13 quantifiziert.

In [ ]:
# =============================================================================
# Zelle 09 – Stemming (PorterStemmer)
# =============================================================================

from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

def apply_stemming(tokens):
    return [stemmer.stem(t) for t in tokens]

train["tokens_stemmed"] = train["tokens_no_stopwords"].apply(apply_stemming)

# Kontrolle
print("=== Beispiele ===\n")
for i in [0, 2, 3]:
    print(f"VORHER:  {train['tokens_no_stopwords'].iloc[i]}")
    print(f"NACHHER: {train['tokens_stemmed'].iloc[i]}\n")

# Vokabulargroesse vor/nach Stemming (erster Indikator fuer Zelle 12/13)
vocab_before = set(t for tokens in train["tokens_no_stopwords"] for t in tokens)
vocab_after = set(t for tokens in train["tokens_stemmed"] for t in tokens)
print(f"Vokabulargroesse vor Stemming: {len(vocab_before)}")
print(f"Vokabulargroesse nach Stemming: {len(vocab_after)}")
print(f"Reduktion: {(1 - len(vocab_after)/len(vocab_before))*100:.1f}%")

In [ ]:
# =============================================================================
# Zelle 10 – Lemmatisierung (WordNetLemmatizer)
# =============================================================================

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def apply_lemmatization(tokens):
    return [lemmatizer.lemmatize(t) for t in tokens]

train["tokens_lemmatized"] = train["tokens_no_stopwords"].apply(apply_lemmatization)

# Kontrolle
print("=== Beispiele ===\n")
for i in [0, 2, 3]:
    print(f"VORHER:  {train['tokens_no_stopwords'].iloc[i]}")
    print(f"NACHHER: {train['tokens_lemmatized'].iloc[i]}\n")

vocab_lemmatized = set(t for tokens in train["tokens_lemmatized"] for t in tokens)
print(f"Vokabulargroesse nach Lemmatisierung: {len(vocab_lemmatized)}")
print(f"Reduktion ggue. vorher: {(1 - len(vocab_lemmatized)/len(vocab_before))*100:.1f}%")

## Erkenntnis: Stemming vs. Lemmatisierung

| Methode | Vokabulargroesse | Reduktion | Wortform |
|---|---|---|---|
| Stemming (Porter) | 12.850 | -18,8% | Oft keine echten Woerter (z. B. "earthquak", "evacu") |
| Lemmatisierung (WordNet) | 14.590 | -7,8% | Ueberwiegend echte Woerter (z. B. "earthquake", "evacuation") |

**Stemming reduziert aggressiver**, da rein regelbasiertes Suffix-
Abschneiden mehrere unterschiedliche Wortformen oft auf denselben
(haeufig kuenstlichen) Stamm zusammenfuehrt.

**Lemmatisierung ist konservativer, aber interpretierbarer** — Woerter
bleiben lesbar, was z. B. fuer die Chi2-Wortanalyse (Notebook 01) oder
spaetere Feature-Importance-Auswertungen (Phase 6) einen klaren Vorteil
bietet.

**Beobachtete Schwaeche der Lemmatisierung:** "us" wurde faelschlich zu
"u" lemmatisiert — WordNetLemmatizer nimmt ohne explizites POS-Tagging
standardmaessig an, jedes Wort sei ein Substantiv, und versucht es zu
singularisieren. Bestaetigt empirisch die in Zelle 10 erwartete
Einschraenkung (fehlende Wortart-Erkennung).

**Entscheidung:** Beide Varianten bleiben als separate Spalten erhalten
(tokens_stemmed, tokens_lemmatized) sowie die unveraenderte Version
(tokens_no_stopwords). Welche Variante tatsaechlich die beste Modell-
Performance liefert, wird NICHT hier entschieden, sondern in Phase 4 als
Ablationsstudie empirisch getestet (gleiches Modell, drei Text-Varianten,
F1-Vergleich per Cross-Validation) — konsistent mit unserem Prinzip
"vorlaeufiger Indikator hier, Beweis erst bei tatsaechlichem Modelltest".

In [ ]:
# =============================================================================
# Zelle 12 – Vorlaeufige Indikatoren: Vokabulargroesse & Entropie ueber alle
# Preprocessing-Stufen hinweg. WICHTIG: Dies ist KEIN Beweis fuer "besser",
# nur ein Proxy. Der eigentliche Test folgt in Phase 4 (Ablationsstudie).
# =============================================================================

import re
from scipy.stats import entropy as scipy_entropy
from collections import Counter

def simple_tokenize_raw(text):
    return re.findall(r"\b\w+\b", text.lower())

# Fuenf Stufen im Vergleich
stages = {
    "Roh (unbereinigt)": train["text"].apply(simple_tokenize_raw),
    "Bereinigt (URL/Mojibake/Zahlen)": train["tokens"],
    "Ohne Stopwords": train["tokens_no_stopwords"],
    "Stemmed": train["tokens_stemmed"],
    "Lemmatized": train["tokens_lemmatized"],
}

results = []
for stage_name, token_series in stages.items():
    all_tokens = [t for tokens in token_series for t in tokens]
    vocab_size = len(set(all_tokens))
    freq = Counter(all_tokens)
    probs = np.array(list(freq.values())) / sum(freq.values())
    ent = scipy_entropy(probs, base=2)
    results.append({"stage": stage_name, "vocab_size": vocab_size, "entropy_bits": round(ent, 3)})

stage_comparison = pd.DataFrame(results)
print(stage_comparison)
stage_comparison.to_csv("../reports/tables/02_preprocessing_stage_comparison.csv", index=False)

# Plot: Vokabulargroesse ueber Stufen
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(stage_comparison["stage"], stage_comparison["vocab_size"], color=COLOR_GOLD)
axes[0].set_title("Vokabulargroesse je Preprocessing-Stufe")
axes[0].set_ylabel("Eindeutige Tokens")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(stage_comparison["stage"], stage_comparison["entropy_bits"], color=COLOR_BLUE)
axes[1].set_title("Entropie der Token-Verteilung")
axes[1].set_ylabel("Entropie (Bits)")
axes[1].tick_params(axis="x", rotation=45)

fig.tight_layout()
save_figure(fig, "../reports/figures/02_preprocessing_stage_comparison.png")
plt.show()

In [ ]:
# =============================================================================
# Zelle 12b – Effektive Vokabulargroesse (2^Entropie) zur besseren Interpretation
# =============================================================================

stage_comparison["effective_vocab"] = (2 ** stage_comparison["entropy_bits"]).round(0).astype(int)
stage_comparison["utilization_pct"] = (stage_comparison["effective_vocab"] / stage_comparison["vocab_size"] * 100).round(1)

print(stage_comparison)
stage_comparison.to_csv("../reports/tables/02_preprocessing_stage_comparison.csv", index=False)

## Erkenntnis: Vorlaeufige Aufarbeitungs-Indikatoren

| Stufe | Vokabular | Entropie (Bit) | Effektive Groesse | Ausnutzung |
|---|---|---|---|---|
| Roh | 21.570 | 10,79 | 1.769 | 8,2% |
| Bereinigt (URL/Mojibake/Zahlen) | 15.968 | 10,89 | 1.903 | 11,9% |
| Ohne Stopwords | 15.821 | 12,17 | 4.592 | 29,0% |
| Stemmed | 12.850 | 11,67 | 3.249 | 25,3% |
| Lemmatized | 14.590 | 11,96 | 3.973 | 27,2% |

**Kernerkenntnis:** Trotz 21.570 unterschiedlicher Woerter im Rohtext
verhaelt sich die Verteilung effektiv nur wie ~1.769 gleich-haeufige
Woerter (8,2% Ausnutzung) — wenige dominante Stopwords (the, a, to...)
konzentrieren den Grossteil der Vorkommen.

**Stopword-Entfernung hat den groessten Einfluss auf die Verteilungs-
Gleichmaessigkeit** (Entropie +1,38 Bit, Ausnutzung 8,2% -> 29,0%),
obwohl die Vokabulargroesse dabei kaum sinkt (21.570 -> 15.821) — Anzahl
unterschiedlicher Woerter und Gleichmaessigkeit ihrer Verteilung sind
zwei unabhaengige Dimensionen.

**Stemming reduziert die Vokabulargroesse am staerksten** (12.850,
-18,8% ggue. "Ohne Stopwords"), Lemmatisierung ist konservativer
(14.590, -7,8%), liefert aber lesbare Wortformen.

**Wichtige Einschraenkung:** Diese Kennzahlen sind Indikatoren fuer
Verteilungseigenschaften, KEIN Beweis fuer bessere Modell-Performance.
Der belastbare Test folgt in Phase 4 als Ablationsstudie: gleiches
Modell (TF-IDF + LogReg), verschiedene Text-Varianten (roh, bereinigt,
stemmed, lemmatized), F1-Vergleich per Cross-Validation.

In [ ]:
# =============================================================================
# Zelle 14 – Speichern: bereinigter & vorverarbeiteter Datensatz fuer Phase 3
# =============================================================================
# Token-Listen werden zu leerzeichengetrennten Strings (direkt nutzbar fuer
# sklearn-Vectorizer in Phase 3, kein Parsing beim Wiedereinlesen noetig).

train["text_no_stopwords"] = train["tokens_no_stopwords"].apply(lambda t: " ".join(t))
train["text_stemmed"] = train["tokens_stemmed"].apply(lambda t: " ".join(t))
train["text_lemmatized"] = train["tokens_lemmatized"].apply(lambda t: " ".join(t))

# Finale Spaltenauswahl: Original, alle Preprocessing-Varianten, Features, Target
columns_to_save = [
    "id", "keyword", "location", "target",
    "text", "text_clean", "text_no_stopwords", "text_stemmed", "text_lemmatized",
    "char_count", "word_count", "has_url", "n_mentions", "n_hashtags",
    "has_mention", "has_hashtag", "has_encoding_artifact",
]

train_preprocessed = train[columns_to_save].copy()

print("Shape:", train_preprocessed.shape)
print("Spalten:", list(train_preprocessed.columns))
print("\nBeispielzeile:")
print(train_preprocessed[["text", "text_clean", "text_no_stopwords", "text_stemmed", "text_lemmatized"]].iloc[2])

train_preprocessed.to_csv("../data/processed/train_preprocessed.csv", index=False)
print(f"\nGespeichert: data/processed/train_preprocessed.csv ({len(train_preprocessed)} Zeilen)")

# Abschluss: Notebook 02 — Preprocessing

## Datensatz-Status

- **Eingelesen:** `data/processed/train_clean.csv` (7.434 Zeilen, aus Notebook 01)
- **Zusaetzliche Bereinigung (text_clean-Basis):** 633 Zeilen entfernt
  (196 Konflikt-Zeilen + 437 harmlose Duplikate nach Normalisierung)
- **Gespeichert:** `data/processed/train_preprocessed.csv` (6.801 Zeilen, 17 Spalten)
- **Gesamtbereinigung ueber beide Notebooks:** 7.613 → 6.801 (-812 Zeilen, -10,7%)

## Gesamtbild der Bereinigung

| Schritt | Zeilen | Entfernt |
|---|---|---|
| Rohdaten (train.csv) | 7.613 | — |
| Notebook 01 (Roh-Text-Duplikate/Konflikte) | 7.434 | 179 |
| Notebook 02 (text_clean-Duplikate/Konflikte) | 6.801 | 633 |
| Gesamt entfernt | | 812 (10,7%) |

**Begruendung:** Qualitaet vor Quantitaet. Widersprueche im Datensatz
erhoehen den Bayes-Fehler (theoretische Untergrenze des Modellfehlers)
unabhaengig vom Modell. Harmlose Duplikate erzeugen bei K-Fold CV
Data-Leakage-Risiko. Datenmenge (6.801) bleibt ausreichend fuer
klassische Modelle (Phase 4).

**Klassenverteilung nach finaler Bereinigung:** 59,4% kein Disaster /
40,6% Disaster (stabil ggue. 57%/43% vor Bereinigung).

## Vier parallele Text-Varianten

text_clean, text_no_stopwords, text_stemmed, text_lemmatized —
ermoeglichen Ablationsstudie in Phase 4.

## Offene Punkte / Bonus-Kandidaten (Phase 8)

- **Datensatz-Variante als Hyperparameter:** Vergleich Modell auf
  bereinigtem Datensatz (6.801) vs. originalem Datensatz (7.613) —
  bei maechtigen Modellen (z. B. DistilBERT) koennte mehr Daten trotz
  Rauschen netto vorteilhaft sein
- **Label Smoothing / Mehrheits-Voting** als Alternative zum harten
  Entfernen von Konflikt-Zeilen (State-of-the-Art bei neuronalen Netzen)
- Subword-Tokenisierung (BPE/WordPiece) fuer Transformer-Modelle

## Konsequenzen fuer Phase 3 (Vektorisierung)

- Alle vier Text-Varianten stehen bereit, koennen parallel vektorisiert
  werden — Ablationsstudie folgt in Phase 4
- keyword als Feature-Kandidat (Notebook 01)
- location nicht verwendet (Notebook 01)